In [7]:
from pathlib import Path
import re
import pandas as pd

BASE_PATH = Path(
    "/Users/manalaltammami/Desktop/twain_child_corpus"
)

text_path = BASE_PATH / "Tom_Sawyer_narrative.txt"
tom_text = text_path.read_text(encoding="utf-8")

# Same word-token rule used in the updated analyses.
TOKEN_PATTERN = re.compile(r"[a-z]+(?:['-][a-z]+)*")

def tokenize(text):
    return TOKEN_PATTERN.findall(
        text.lower().replace("’", "'")
    )

total_tokens = len(tokenize(tom_text))

assert total_tokens == 70518, (
    f"Expected 70,518 tokens; found {total_tokens:,}. "
    "Check that you loaded the final narrative file."
)

print(f"Tom Sawyer: {total_tokens:,} word tokens")
print(f"Opening quotes: {tom_text.count('“'):,}")
print(f"Closing quotes: {tom_text.count('”'):,}")


Tom Sawyer: 70,518 word tokens
Opening quotes: 1,522
Closing quotes: 1,519


In [9]:
from datetime import datetime
import hashlib

passages = []
start = None
boundary_note = ""

def add_passage(start, end, note):
    passage = tom_text[start:end]

    if not tokenize(passage):
        return

    passages.append({
        "Passage ID": f"Tom_Q{len(passages) + 1:04d}",
        "Start": start,
        "End": end,
        "Word tokens": len(tokenize(passage)),
        "Before": tom_text[max(0, start - 700):start],
        "Quoted passage": passage,
        "After": tom_text[end:end + 700],
        "Boundary note": note,
        "Quotation type": "",
        "Speaker": "",
        "Review notes": "",
    })

for position, character in enumerate(tom_text):

    if character == "“":
        if start is not None:
            # A repeated opening mark may continue the same
            # speaker's speech across paragraphs.
            add_passage(
                start,
                position,
                "Repeated opening quote: review continuation"
            )
            boundary_note = (
                "Starts after repeated opening quote: "
                "review continuation"
            )
        else:
            boundary_note = ""

        start = position + 1

    elif character == "”":
        if start is None:
            raise ValueError(
                f"Closing quote without opening at {position}. "
                "Check this boundary before continuing."
            )

        add_passage(start, position, boundary_note)
        start = None
        boundary_note = ""

if start is not None:
    raise ValueError(
        "An opening quote remains unclosed at the end. "
        "Check this boundary before continuing."
    )

tom_passages = pd.DataFrame(passages)

# Record the exact source text used for this extraction.
tom_passages["Source SHA256"] = hashlib.sha256(
    tom_text.encode("utf-8")
).hexdigest()

review_folder = BASE_PATH / "becky_dialogue_review"
review_folder.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
review_path = (
    review_folder
    / f"Tom_all_quoted_passages_{timestamp}.csv"
)

tom_passages.to_csv(review_path, index=False)

print(f"Quoted passages: {len(tom_passages):,}")
print(
    "Quoted word tokens:",
    f"{tom_passages['Word tokens'].sum():,}"
)
print(f"Saved to: {review_path}")
print("\nSpeaker labels are blank for review.")


Quoted passages: 1,522
Quoted word tokens: 22,613
Saved to: /Users/manalaltammami/Desktop/twain_child_corpus/becky_dialogue_review/Tom_all_quoted_passages_20261104_203944_065734.csv

Speaker labels are blank for review.


In [84]:
# Load the reviewed file
review_path = (
    Path.home()
    / "Downloads"
    / "Tom_passages_REVIEWED_Mary_Polly.csv"
)
reviewed_passages = pd.read_csv(review_path).fillna("")

# Confirm that the review belongs to the loaded text
source_hash = hashlib.sha256(
    tom_text.encode("utf-8")
).hexdigest()

if not reviewed_passages["Source SHA256"].eq(source_hash).all():
    raise ValueError("The review file belongs to a different text.")

# Read the inclusion labels safely
inclusion = (
    reviewed_passages["Speech included"]
    .astype(str)
    .str.strip()
    .str.lower()
)

if not inclusion.isin(["true", "false"]).all():
    raise ValueError("Some speech-inclusion labels are missing or invalid.")

spoken_passages = reviewed_passages.loc[
    inclusion.eq("true")
].copy()

# Recalculate words, excluding bracketed stage directions
spoken_passages["Calculated words"] = (
    spoken_passages["Quoted passage"]
    .map(lambda text: len(tokenize(
        re.sub(r"\[[^\]]*\]", " ", text)
    )))
)

total_spoken_words = spoken_passages["Calculated words"].sum()

# All retained speech contributes to the denominator,
# including joint or uncertain speaker attributions.
character_summary = (
    spoken_passages
    .groupby("Speaker")
    .agg(
        Quotation_passages=("Passage ID", "size"),
        Dialogue_words=("Calculated words", "sum")
    )
)

character_summary["Share (%)"] = (
    character_summary["Dialogue_words"]
    / total_spoken_words
    * 100
)

comparison = character_summary.loc[
    ["Becky Thatcher", "Tom Sawyer", "Huck Finn"]
]

print(f"Total retained spoken words: {total_spoken_words:,}")
print()
print(comparison.round(2).to_string())

print(
    "\nQuotation passages are not speech turns. "
    "Self-address and nonspoken quotations are excluded."
)

Total retained spoken words: 21,353

                Quotation_passages  Dialogue_words  Share (%)
Speaker                                                      
Becky Thatcher                  75             750       3.51
Tom Sawyer                     528            8127      38.06
Huck Finn                      276            4569      21.40

Quotation passages are not speech turns. Self-address and nonspoken quotations are excluded.


In [86]:
# Show Becky, Mary, and Aunt Polly using existing speaker labels.
labels = character_summary.index.astype(str)

selected = character_summary.loc[
    labels.str.contains(
        r"Becky|Mary|Polly", case=False, regex=True
    )
].copy()

selected["Words per passage"] = (
    selected["Dialogue_words"]
    / selected["Quotation_passages"]
)

print(selected.round(2).to_string())

if selected.empty:
    print("\nAvailable speaker labels:")
    print(character_summary.index.tolist())

                Quotation_passages  Dialogue_words  Share (%)  Words per passage
Speaker                                                                         
Aunt Polly                      93            1735       8.13              18.66
Becky Thatcher                  75             750       3.51              10.00
Mary                            15             135       0.63               9.00


In [90]:
comparison = character_summary.loc[
    ["Becky Thatcher", "Mary", "Aunt Polly",
     "Tom Sawyer", "Huck Finn"]
]

print(comparison.round(2).to_string())

                Quotation_passages  Dialogue_words  Share (%)
Speaker                                                      
Becky Thatcher                  75             750       3.51
Mary                            15             135       0.63
Aunt Polly                      93            1735       8.13
Tom Sawyer                     528            8127      38.06
Huck Finn                      276            4569      21.40


In [96]:
from pathlib import Path
import pandas as pd

review_path = (
    Path.home()
    / "Downloads"
    / "Vernacular_218_Context_Review"
    / "dialect_review_218_CONTEXT_REVIEWED.csv"
)

print("Loading:", review_path)
review = pd.read_csv(review_path, keep_default_na=False)

Loading: /Users/manalaltammami/Downloads/Vernacular_218_Context_Review/dialect_review_218_CONTEXT_REVIEWED.csv


In [98]:
from pathlib import Path
import re
import pandas as pd

# Load the reviewed vernacular occurrences.
review_path = (
    Path.home()
    / "Downloads"
    / "Vernacular_218_Context_Review"
    / "dialect_review_218_CONTEXT_REVIEWED.csv"
)

review = pd.read_csv(review_path, keep_default_na=False)
print("Loading:", review_path)

matches = review.loc[
    review["File"].eq("Tom_Sawyer_narrative.txt")
    & review["Decision"].str.strip().eq("Include")
].copy()

characters = [
    "Becky Thatcher",
    "Mary",
    "Aunt Polly",
    "Tom Sawyer",
    "Huck Finn",
]

# Check that the updated speaker file is loaded.
missing = [
    name for name in characters
    if name not in spoken_passages["Speaker"].values
]

if missing:
    raise ValueError(
        "Load Tom_passages_REVIEWED_Mary_Polly.csv first. "
        "Missing speaker labels: " + ", ".join(missing)
    )

# Check occurrence positions against the source text.
for _, match in matches.iterrows():
    start = int(match["Start"])
    end = int(match["End"])

    if tom_text[start:end] != match["Matched wording"]:
        raise ValueError(
            f"Source text mismatch at position {start}."
        )

results = []

for character in characters:
    passages = spoken_passages.loc[
        spoken_passages["Speaker"].eq(character)
    ]

    words = int(passages["Calculated words"].sum())
    counts = {"dialect": 0, "colloquial": 0}

    for _, match in matches.iterrows():
        start = int(match["Start"])
        end = int(match["End"])

        containing = passages.loc[
            passages["Start"].astype(int).le(start)
            & passages["End"].astype(int).ge(end)
        ]

        if containing.empty:
            continue

        if len(containing) > 1:
            raise ValueError("Overlapping attributed passages.")

        passage = containing.iloc[0]
        passage_start = int(passage["Start"])
        passage_end = int(passage["End"])

        # Exclude bracketed narrative directions.
        brackets = re.finditer(
            r"\[[^\]]*\]",
            tom_text[passage_start:passage_end]
        )

        inside_brackets = any(
            passage_start + bracket.start() <= start
            and end <= passage_start + bracket.end()
            for bracket in brackets
        )

        if inside_brackets:
            continue

        categories = {
            value.strip().lower()
            for value in match["Category"].split(";")
        }

        for category in counts:
            if category in categories:
                counts[category] += 1

    results.append({
        "Character": character,
        "Spoken words": words,
        "Dialect matches": counts["dialect"],
        "Dialect per 10k": round(
            counts["dialect"] / words * 10000, 2
        ) if words else None,
        "Colloquial matches": counts["colloquial"],
        "Colloquial per 10k": round(
            counts["colloquial"] / words * 10000, 2
        ) if words else None,
    })

character_vernacular = pd.DataFrame(results)

print()
print(character_vernacular.to_string(index=False))
print(
    "\nRates use reviewed inventory matches. "
    "Ain’t is excluded from this inventory. "
    "Mary’s sample contains only 135 spoken words."
)

Loading: /Users/manalaltammami/Downloads/Vernacular_218_Context_Review/dialect_review_218_CONTEXT_REVIEWED.csv

     Character  Spoken words  Dialect matches  Dialect per 10k  Colloquial matches  Colloquial per 10k
Becky Thatcher           750                0             0.00                   3               40.00
          Mary           135                0             0.00                   0                0.00
    Aunt Polly          1735                7            40.35                   6               34.58
    Tom Sawyer          8127               18            22.15                  49               60.29
     Huck Finn          4569               42            91.92                  32               70.04

Rates use reviewed inventory matches. Ain’t is excluded from this inventory. Mary’s sample contains only 135 spoken words.
